> 在线讲义：[第 14 章 · 主流 VLA 模型全景：架构、关键改进、优劣势](https://wikieden.github.io/vla-from-scratch/lessons/14-vla-landscape.html) · 本书《VLA 从零到 π0》

# 第 14 章 · 主流 VLA 模型全景：架构、关键改进、优劣势

配套讲义：仓库里 `vla-deep-dive/notes/lessons/14-vla-landscape.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python + NumPy，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 写法 | 跑出来 |
|---|---|---|
| 第 2 步 | `MODELS`、`group_by_head(MODELS)` | 离散 token：RT-2、OpenVLA、π0-FAST；flow 头：π0、GR00T N1、SmolVLA 等 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
MODELS = {
    "ACT": ("一次出一段（CVAE）", "没有 VLM", 2023),
    "Diffusion Policy": ("扩散头", "没有 VLM", 2023),
    "RT-2": ("离散 token", "同一个模型念词", 2023),
    "OpenVLA": ("离散 token", "同一个模型念词", 2024),
    "π0-FAST": ("离散 token（FAST）", "同一个模型念词", 2025),
    "OpenVLA-OFT": ("连续回归（L1）", "同一个模型，并行解码", 2025),
    "Octo": ("扩散头", "小 Transformer 主干", 2024),
    "RDT-1B": ("扩散头", "扩散 Transformer", 2024),
    "π0": ("flow 头", "两套权重共享 attention", 2024),
    "CogACT": ("扩散头", "VLM 认知特征 → 扩散模块", 2024),
    "GR00T N1": ("flow 头", "DiT cross-attention 读 VLM 第 12 层", 2025),
    "SmolVLA": ("flow 头", "小 VLM 前一半层 + 动作专家", 2025),
}

**函数 `group_by_head`**：按"动作怎么出"把模型分组。

In [ ]:
def group_by_head(models):
    """按"动作怎么出"把模型分组。"""
    groups = {}
    for name, (head, _, _) in models.items():
        key = "离散 token" if head.startswith("离散") else head
        groups.setdefault(key, []).append(name)
    return groups

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 按动作怎么出排一排")
for head, names in group_by_head(MODELS).items():
    print(f"  {head}：{', '.join(names)}")
flows = [n for n, (h, *_) in MODELS.items() if h in ("flow 头", "扩散头")]
print(f"  扩散 / flow 头一共 {len(flows)} 个")

## 3. 代码里才看得到的

1. `MODELS` 只收了第 2 步 A–D 组的 12 个模型；E、F 两组讲的是训练配方和数据，不按动作头分。
2. 分组时把"离散 token（FAST）"也归进离散 token：FAST 改的是怎么切词，不是动作头。

## 4. 自己改一改

- 把第 15–21 章的新模型加进 `MODELS`，看各组的数量怎么变。
- 改成按"大脑小脑怎么连"分组。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 看车看三样：发动机、变速箱、跑过的路。
> 念词有常识但慢，回归最快但不分峰，flow 多峰又顺，视频吃得最多。
> 没有全赢的模型，只有对得上的选择。